# Task 5 - Root Cause Investigation

Where is the deterioration concentrated, how much does each segment **contribute**
to it, and is that contribution a **mix** effect or a **within-segment** effect?

Ranking segments cannot answer this. The decomposition below is exact:

    delta_c  =  sum_i (w_i1 - w_i0) * c_i0     <- mix: the weights moved
             +  sum_i  w_i1 * (c_i1 - c_i0)    <- within: the economics moved

Both terms sum to the observed change with zero residual. That is the test that
the attribution is complete rather than a plausible story.

Put this notebook in `05_profitability_model/`. It reads the fact table Task 4 wrote.

In [19]:
from pathlib import Path
"""Task 5 - root cause investigation.

Two questions the brief asks that ranking cannot answer:

  1. How much of the total deterioration does each segment CONTRIBUTE?
  2. Is that contribution a MIX effect (the segment's weight moved) or a
     WITHIN-SEGMENT effect (the segment's own economics got worse)?

The decomposition is exact. For segments i, between period 0 and period 1:

    delta_c = sum_i (w_i1 - w_i0) * c_i0      <- mix
            + sum_i  w_i1 * (c_i1 - c_i0)     <- within

The two terms sum to the observed change with no residual.
"""
import numpy as np
import pandas as pd

DIMENSIONS = [
    ("merchant_category", "Merchant category"),
    ("merchant_country", "Merchant country"),
    ("channel", "Payment channel"),
    ("provider", "Provider"),
    ("customer_segment", "Customer segment"),
    ("merchant_risk_band", "Merchant risk band"),
    ("merchant_pricing_plan", "Pricing plan"),
    ("ticket_band", "Ticket size band"),
    ("cohort", "Category x country cohort"),
]

TICKET_EDGES = [-np.inf, 25, 50, 100, 250, 500, np.inf]
TICKET_LABELS = ["under $25", "$25-50", "$50-100", "$100-250", "$250-500", "over $500"]

CANDIDATES = [
    Path("model_output/fact_transaction.csv"),
    Path("../04_profitability_model/model_output/fact_transaction.csv"),
    Path("../05_profitability_model/model_output/fact_transaction.csv"),
]
FACT = next((p for p in CANDIDATES if p.exists()), None)
if FACT is None:
    raise FileNotFoundError(
        "fact_transaction.csv not found. Run the Task 4 notebook first - it writes "
        "the fact table this analysis consumes.")

OUT_FILE = "05_root_cause_analysis.xlsx"

fact = pd.read_csv(FACT, parse_dates=["txn_date"])
print(f"{len(fact):,} rows from {FACT}")

260,287 rows from ..\04_profitability_model\model_output\fact_transaction.csv


## Prepare

In [20]:
def prepare(fact):
    """Successful transactions only, with the derived grouping columns."""
    f = fact[fact.is_success & ~fact.dq_quarantined].copy()
    # rows with a missing merchant are real payments - keep them as their own
    # segment so the decomposition stays exact instead of silently dropping them
    for col in ["merchant_category", "merchant_country", "merchant_risk_band",
                "merchant_pricing_plan", "customer_segment", "channel", "provider"]:
        f[col] = f[col].fillna("(unknown)")
    f["ticket_band"] = pd.cut(f.gross_usd, TICKET_EDGES, labels=TICKET_LABELS)
    f["cohort"] = f.merchant_category.astype(str) + " / " + f.merchant_country.astype(str)
    f["contribution_per_txn"] = f.contribution_usd
    return f

f = prepare(fact)
FIRST, LAST = f.txn_month.min(), f.txn_month.max()
print(f"{len(f):,} successful transactions, {FIRST} to {LAST}")

233,208 successful transactions, 2025-01 to 2025-12


## Headline bridge

Which component of the unit economics moved, before asking which segment.

In [21]:
def headline(f, first, last):
    """Bridge the executive KPI from the opening to the closing month."""
    a, b = f[f.txn_month == first], f[f.txn_month == last]
    comp = []
    for label, col, sign in [
        ("Merchant revenue per successful txn", "merchant_revenue_usd", +1),
        ("Processing cost per successful txn", "processing_cost_usd", -1),
        ("Chargeback cost per successful txn", "chargeback_cost_usd", -1),
    ]:
        v0, v1 = a[col].mean(), b[col].mean()
        comp.append((label, round(v0, 4), round(v1, 4), round(v1 - v0, 4),
                     round(sign * (v1 - v0), 4)))
    out = pd.DataFrame(comp, columns=["component", first, last, "change",
                                      "effect_on_contribution"])
    c0, c1 = a.contribution_usd.mean(), b.contribution_usd.mean()
    out.loc[len(out)] = ["Contribution per successful txn", round(c0, 4), round(c1, 4),
                         round(c1 - c0, 4), round(c1 - c0, 4)]
    return out

bridge = headline(f, FIRST, LAST)
bridge

,component,2025-01,2025-12,change,effect_on_contribution
0,Merchant revenue per successful txn,2.9410,2.1972,-0.7437,-0.7437
1,Processing cost per successful txn,1.0762,0.9672,-0.1090,0.1090
2,Chargeback cost per successful txn,0.2515,0.1495,-0.1020,0.1020
3,Contribution per successful txn,1.6132,1.0805,-0.5328,-0.5328


## Mix versus within, across every dimension

The same change, decomposed nine different ways. The dimension with the highest
mix share is the one that best explains what happened. A residual of zero on
every row proves nothing has been left out.

In [22]:
def _effects(f, dim, first, last):
    """Unrounded mix / within series. Rounding is for display only."""
    a, b = f[f.txn_month == first], f[f.txn_month == last]
    idx = sorted(set(a[dim].dropna().astype(str)) | set(b[dim].dropna().astype(str)))

    w0 = (a[dim].astype(str).value_counts(normalize=True).reindex(idx).fillna(0))
    w1 = (b[dim].astype(str).value_counts(normalize=True).reindex(idx).fillna(0))
    c0 = a.groupby(a[dim].astype(str)).contribution_usd.mean().reindex(idx).fillna(0)
    c1 = b.groupby(b[dim].astype(str)).contribution_usd.mean().reindex(idx).fillna(0)

    return idx, w0, w1, c0, c1, (w1 - w0) * c0, w1 * (c1 - c0)


def decompose(f, dim, first, last):
    """Exact mix / within split of the change in contribution per successful txn."""
    idx, w0, w1, c0, c1, mix, within = _effects(f, dim, first, last)
    out = pd.DataFrame({
        "segment": idx,
        "share_open": w0.to_numpy().round(4),
        "share_close": w1.to_numpy().round(4),
        "share_change": (w1 - w0).to_numpy().round(4),
        "contrib_per_txn_open": c0.to_numpy().round(4),
        "contrib_per_txn_close": c1.to_numpy().round(4),
        "mix_effect": mix.to_numpy().round(4),
        "within_effect": within.to_numpy().round(4),
    })
    out["total_effect"] = (out.mix_effect + out.within_effect).round(4)
    total = out.total_effect.sum()
    out["pct_of_total_change"] = (100 * out.total_effect / total).round(1) if total else 0.0
    return out.sort_values("total_effect").reset_index(drop=True)


def decomposition_summary(f, first, last):
    """One row per dimension: how much of the change is mix vs within."""
    actual = (f[f.txn_month == last].contribution_usd.mean()
              - f[f.txn_month == first].contribution_usd.mean())
    rows = []
    for dim, label in DIMENSIONS:
        _, _, _, _, _, mix_s, within_s = _effects(f, dim, first, last)
        mix, within = mix_s.sum(), within_s.sum()
        rows.append((label, dim, round(mix, 4), round(within, 4),
                     round(mix + within, 4), round(actual, 4),
                     round(100 * mix / (mix + within), 1) if (mix + within) else 0,
                     round(abs((mix + within) - actual), 6)))
    return pd.DataFrame(rows, columns=[
        "dimension", "column", "mix_effect", "within_effect", "sum_of_effects",
        "actual_change", "mix_share_pct", "residual"])

summary = decomposition_summary(f, FIRST, LAST)
summary

,dimension,column,mix_effect,within_effect,sum_of_effects,actual_change,mix_share_pct,residual
0,Merchant category,merchant_category,-0.3574,-0.1754,-0.5328,-0.5328,67.1,0.0
1,Merchant country,merchant_country,0.0539,-0.5866,-0.5328,-0.5328,-10.1,0.0
2,Payment channel,channel,0.0222,-0.5550,-0.5328,-0.5328,-4.2,0.0
3,Provider,provider,0.0716,-0.6044,-0.5328,-0.5328,-13.4,0.0
4,Customer segment,customer_segment,-0.0455,-0.4872,-0.5328,-0.5328,8.5,0.0
5,Merchant risk band,merchant_risk_band,-0.0091,-0.5237,-0.5328,-0.5328,1.7,0.0
6,Pricing plan,merchant_pricing_plan,-0.2299,-0.3029,-0.5328,-0.5328,43.1,0.0
7,Ticket size band,ticket_band,-0.4719,-0.0609,-0.5328,-0.5328,88.6,0.0
8,Category x country cohort,cohort,-0.4272,-0.1056,-0.5328,-0.5328,80.2,0.0


## Segment attribution

Contribution to the total change, not a ranking. A segment can matter because
its weight moved, because its economics moved, or both.

In [23]:
attribution = {}
for dim, label in DIMENSIONS:
    attribution[label] = decompose(f, dim, FIRST, LAST)

for label in ["Ticket size band", "Category x country cohort", "Provider"]:
    print(f"\n=== {label} ===")
    print(attribution[label].to_string(index=False))


=== Ticket size band ===
  segment  share_open  share_close  share_change  contrib_per_txn_open  contrib_per_txn_close  mix_effect  within_effect  total_effect  pct_of_total_change
over $500      0.1016       0.0696       -0.0320                7.4383                 7.8564     -0.2379         0.0291       -0.2088                 39.2
 $100-250      0.2383       0.1663       -0.0719                1.5492                 1.3879     -0.1115        -0.0268       -0.1383                 26.0
 $250-500      0.1045       0.0740       -0.0305                3.0270                 2.8925     -0.0924        -0.0099       -0.1023                 19.2
under $25      0.1504       0.3367        0.1863                0.0257                -0.1629      0.0048        -0.0635       -0.0587                 11.0
  $50-100      0.2318       0.1720       -0.0598                0.6050                 0.6311     -0.0362         0.0045       -0.0317                  6.0
   $25-50      0.1734       0.1813    

## Driver bridge

The same total, attributed to named drivers instead of segments, and split into
operational, commercial and risk. The residual must be zero.

In [24]:
def monthly_by(f, dim):
    """Monthly series of weight and contribution per txn for every segment."""
    g = (f.groupby(["txn_month", f[dim].astype(str)])
           .agg(successful=("is_success", "sum"),
                contribution_usd=("contribution_usd", "sum"))
           .reset_index().rename(columns={dim: "segment", "level_1": "segment"}))
    g.columns = ["txn_month", "segment", "successful", "contribution_usd"]
    tot = g.groupby("txn_month").successful.transform("sum")
    g["share"] = (g.successful / tot).round(4)
    g["contrib_per_txn"] = (g.contribution_usd / g.successful).round(4)
    return g


def driver_bridge(f, first, last):
    """Attribute the change to named drivers, each measured on its own terms."""
    a, b = f[f.txn_month == first], f[f.txn_month == last]
    rows = []

    # cost side: split into route mix and route rate, holding the other fixed
    prov0 = a.provider.value_counts(normalize=True)
    prov1 = b.provider.value_counts(normalize=True)
    cost0 = a.groupby("provider").processing_cost_usd.mean()
    cost1 = b.groupby("provider").processing_cost_usd.mean()
    idx = sorted(set(prov0.index) | set(prov1.index))
    prov0, prov1 = prov0.reindex(idx).fillna(0), prov1.reindex(idx).fillna(0)
    cost0, cost1 = cost0.reindex(idx).fillna(0), cost1.reindex(idx).fillna(0)

    route_mix = -((prov1 - prov0) * cost0).sum()
    route_rate = -(prov1 * (cost1 - cost0)).sum()
    rows.append(("Routing mix - traffic moved between providers", round(route_mix, 4),
                 "Operational"))
    rows.append(("Provider unit cost - cost per txn within provider", round(route_rate, 4),
                 "Operational"))

    rev_change = b.merchant_revenue_usd.mean() - a.merchant_revenue_usd.mean()
    rows.append(("Merchant revenue per txn", round(rev_change, 4), "Commercial"))

    cb_change = -(b.chargeback_cost_usd.mean() - a.chargeback_cost_usd.mean())
    rows.append(("Chargeback cost per txn", round(cb_change, 4), "Risk"))

    fx_change = b.fx_impact_usd.mean() - a.fx_impact_usd.mean()
    rows.append(("FX translation (memo, already inside revenue)", round(fx_change, 4), "Memo"))

    out = pd.DataFrame(rows, columns=["driver", "effect_on_contribution_per_txn", "type"])
    actual = b.contribution_usd.mean() - a.contribution_usd.mean()
    explained = out[out.type != "Memo"].effect_on_contribution_per_txn.sum()
    out.loc[len(out)] = ["Total explained", round(explained, 4), ""]
    out.loc[len(out)] = ["Actual change", round(actual, 4), ""]
    out.loc[len(out)] = ["Unexplained residual", round(actual - explained, 4), ""]
    return out


def ticket_mix(f):
    """Where the growth actually went - the mechanism behind the mix shift."""
    g = (f.groupby(["txn_month", "ticket_band"], observed=True)
           .agg(successful=("is_success", "sum"),
                contribution_usd=("contribution_usd", "sum"),
                median_ticket_usd=("gross_usd", "median"))
           .reset_index())
    tot = g.groupby("txn_month").successful.transform("sum")
    g["share"] = (g.successful / tot).round(4)
    g["contrib_per_txn"] = (g.contribution_usd / g.successful).round(4)
    g["median_ticket_usd"] = g.median_ticket_usd.round(2)
    return g

drivers = driver_bridge(f, FIRST, LAST)
drivers

,driver,effect_on_contribution_per_txn,type
0,Routing mix - traffic moved between providers,0.0198,Operational
1,Provider unit cost - cost per txn within provider,0.0892,Operational
2,Merchant revenue per txn,-0.7437,Commercial
3,Chargeback cost per txn,0.1020,Risk
4,"FX translation (memo, already inside revenue)",-0.0802,Memo
5,Total explained,-0.5327,
6,Actual change,-0.5328,
7,Unexplained residual,-0.0001,


## Monthly series

In [25]:
tickets = ticket_mix(f)
pivot_share = tickets.pivot(index="txn_month", columns="ticket_band", values="share")
pivot_contrib = tickets.pivot(index="txn_month", columns="ticket_band", values="contrib_per_txn")
print("share of successful transactions by ticket band")
print(pivot_share.to_string())
print("\ncontribution per successful transaction by ticket band")
print(pivot_contrib.to_string())

share of successful transactions by ticket band
ticket_band  under $25  $25-50  $50-100  $100-250  $250-500  over $500
txn_month                                                             
2025-01         0.1504  0.1734   0.2318    0.2383    0.1045     0.1016
2025-02         0.1630  0.1751   0.2247    0.2296    0.1101     0.0975
2025-03         0.1756  0.1805   0.2177    0.2292    0.0988     0.0982
2025-04         0.1941  0.1812   0.2093    0.2260    0.0986     0.0908
2025-05         0.2075  0.1774   0.2143    0.2157    0.0955     0.0895
2025-06         0.2188  0.1806   0.2091    0.2126    0.0953     0.0834
2025-07         0.2385  0.1773   0.2021    0.2032    0.0920     0.0869
2025-08         0.2633  0.1738   0.1976    0.1947    0.0915     0.0791
2025-09         0.2832  0.1771   0.1888    0.1903    0.0834     0.0774
2025-10         0.2996  0.1759   0.1864    0.1821    0.0842     0.0719
2025-11         0.3214  0.1742   0.1778    0.1753    0.0788     0.0725
2025-12         0.3367  0.181

## Write the analysis

In [26]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows

NAVY, ARIAL = "1F3864", "Arial"
thin = Side(style="thin", color="BFBFBF")
box = Border(left=thin, right=thin, top=thin, bottom=thin)


def write_sheet(wb, title, df, widths=None):
    ws = wb.create_sheet(title[:31])
    for row in dataframe_to_rows(df, index=False, header=True):
        ws.append(row)
    for c in range(1, df.shape[1] + 1):
        cell = ws.cell(row=1, column=c)
        cell.font = Font(name=ARIAL, bold=True, size=10, color="FFFFFF")
        cell.fill = PatternFill("solid", fgColor=NAVY)
        cell.alignment = Alignment(vertical="center", wrap_text=True)
        cell.border = box
    ws.row_dimensions[1].height = 30
    for r in range(2, df.shape[0] + 2):
        for c in range(1, df.shape[1] + 1):
            cell = ws.cell(row=r, column=c)
            cell.font = Font(name=ARIAL, size=9)
            cell.alignment = Alignment(vertical="top", wrap_text=True)
            cell.border = box
    for col, w in (widths or {}).items():
        ws.column_dimensions[col].width = w
    ws.freeze_panes = "A2"
    return ws


wb = Workbook()
wb.remove(wb.active)

cover = wb.create_sheet("Cover")
cover.sheet_view.showGridLines = False
a, b = f[f.txn_month == FIRST], f[f.txn_month == LAST]
lines = [
    ("AstraPay - Payment Profitability Diagnostic", 18, True),
    ("Task 5 - Root Cause Investigation", 13, False),
    ("", 10, False),
    (f"Period compared: {FIRST} against {LAST}", 10, False),
    (f"Contribution per successful transaction: "
     f"{a.contribution_usd.mean():.3f} to {b.contribution_usd.mean():.3f} USD", 10, False),
    ("", 10, False),
    ("Method", 11, True),
    ("delta = sum (w1 - w0) * c0   [mix]   +   sum w1 * (c1 - c0)   [within]", 10, False),
    ("The two terms sum to the observed change exactly. Residual is reported per dimension", 10, False),
    ("and must be zero - that is the proof the attribution is complete.", 10, False),
    ("", 10, False),
    ("Reading the tables", 11, True),
    ("mix_effect: the segment's share of volume changed, its own economics did not.", 10, False),
    ("within_effect: the segment's own contribution per transaction changed.", 10, False),
    ("pct_of_total_change: how much of the whole deterioration this segment accounts for.", 10, False),
    ("A negative pct means the segment moved against the trend.", 10, False),
]
for i, (text, size, bold) in enumerate(lines, start=2):
    c = cover.cell(row=i, column=2, value=text)
    c.font = Font(name=ARIAL, size=size, bold=bold, color=NAVY if bold else "000000")
cover.column_dimensions["A"].width = 3
cover.column_dimensions["B"].width = 108

write_sheet(wb, "Headline bridge", bridge, {"A": 38, "B": 14, "C": 14, "D": 12, "E": 22})
write_sheet(wb, "Decomposition summary", summary,
            {"A": 28, "B": 24, "C": 13, "D": 15, "E": 16, "F": 15, "G": 15, "H": 11})
write_sheet(wb, "Driver bridge", drivers, {"A": 52, "B": 30, "C": 14})

for label, df in attribution.items():
    write_sheet(wb, label, df,
                {"A": 30, "B": 12, "C": 12, "D": 13, "E": 20, "F": 20,
                 "G": 12, "H": 14, "I": 13, "J": 19})

write_sheet(wb, "Ticket band monthly", tickets,
            {"A": 12, "B": 14, "C": 12, "D": 18, "E": 18, "F": 10, "G": 16})

wb.save(OUT_FILE)
print("wrote", OUT_FILE, "-", len(wb.sheetnames), "sheets")

wrote 05_root_cause_analysis.xlsx - 14 sheets


## Check

In [27]:
print("residual by dimension (must all be zero):")
print(summary[["dimension", "sum_of_effects", "actual_change", "residual"]].to_string(index=False))
print("\ndriver bridge residual:")
print(drivers.tail(3).to_string(index=False))

residual by dimension (must all be zero):
                dimension  sum_of_effects  actual_change  residual
        Merchant category         -0.5328        -0.5328       0.0
         Merchant country         -0.5328        -0.5328       0.0
          Payment channel         -0.5328        -0.5328       0.0
                 Provider         -0.5328        -0.5328       0.0
         Customer segment         -0.5328        -0.5328       0.0
       Merchant risk band         -0.5328        -0.5328       0.0
             Pricing plan         -0.5328        -0.5328       0.0
         Ticket size band         -0.5328        -0.5328       0.0
Category x country cohort         -0.5328        -0.5328       0.0

driver bridge residual:
              driver  effect_on_contribution_per_txn type
     Total explained                         -0.5327     
       Actual change                         -0.5328     
Unexplained residual                         -0.0001     
